In [ ]:
import copy
import os
import platform
import random
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm.auto import tqdm
import kagglehub

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# Download dataset (wrap in try/except so notebook doesn't crash)
try:
    path = kagglehub.dataset_download("ayyuce/vegetables")
except Exception as e:
    warnings.warn(f"dataset_download failed: {e} — ensure kagglehub is configured.")
    path = './'

DATASET_ROOT = os.path.join(path, "dataset")
IMAGES_DIR = os.path.join(DATASET_ROOT, "images")
LABELS_DIR = os.path.join(DATASET_ROOT, "labels")

# Class mapping: 0..N-1 for object ids (we add +1 for background in targets)
CLASS_NAMES = {
    0: "Lettuce", 1: "Potato", 2: "Carrot", 3: "Onion", 
    4: "Garlic", 5: "Leek", 6: "Broccoli"
}

# Choose reasonable default workers (Windows -> 0 to avoid spawn issues in notebooks)
NUM_WORKERS = 0 if platform.system() == 'Windows' else min(4, os.cpu_count() or 4)


In [ ]:
class VegetableDataset(Dataset):
    def __init__(self, image_files, images_dir, labels_dir, transforms=None):
        self.image_files = image_files
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms

    def __getitem__(self, idx):
        # Load image
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        
        # Load labels
        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.labels_dir, label_name)
        
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    # YOLO: class_id, x_c, y_c, w, h
                    data = list(map(float, line.split()))
                    class_id = int(data[0])
                    x_c, y_c, b_w, b_h = data[1:]
                    
                    # Convert to [xmin, ymin, xmax, ymax]
                    w, h = img.size
                    xmin = (x_c - b_w / 2) * w
                    ymin = (y_c - b_h / 2) * h
                    xmax = (x_c + b_w / 2) * w
                    ymax = (y_c + b_h / 2) * h
                    
                    # Avoid degenerate boxes
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        # Labels in Faster R-CNN are 1-indexed (0 is usually background)
                        labels.append(class_id + 1)

        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_files)

In [ ]:
class PreloadedVegetableDataset(Dataset):
    def __init__(self, image_files, images_dir, labels_dir, transforms=None):
        self.image_files = image_files
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        
        # Lists to store loaded data in RAM
        self.images = []
        self.targets = []

        print(f"Preloading {len(image_files)} images into RAM...")
        
        for idx, img_name in enumerate(tqdm(image_files)):
            # 1. Load Image
            img_path = os.path.join(self.images_dir, img_name)
            img = Image.open(img_path).convert("RGB")
            
            # 2. Load Labels/Boxes
            label_name = os.path.splitext(img_name)[0] + ".txt"
            label_path = os.path.join(self.labels_dir, label_name)
            
            boxes = []
            labels = []
            
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    for line in f:
                        data = list(map(float, line.split()))
                        class_id = int(data[0])
                        x_c, y_c, b_w, b_h = data[1:]
                        
                        w, h = img.size
                        xmin = (x_c - b_w / 2) * w
                        ymin = (y_c - b_h / 2) * h
                        xmax = (x_c + b_w / 2) * w
                        ymax = (y_c + b_h / 2) * h
                        
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(class_id + 1)

            # Convert to tensors immediately to save space and time later
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "image_id": torch.tensor([idx])
            }
            
            # Store the PIL image and the target dictionary
            self.images.append(img)
            self.targets.append(target)

    def __getitem__(self, idx):
        # Retrieve from RAM
        img = self.images[idx]
        target = self.targets[idx]
        
        # Apply transforms on the fly (important for Data Augmentation)
        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.images)

In [ ]:
# --- CONFIGURATION ---
BATCH_SIZE = 4
PRELOAD = True  # Set to True to load all images into RAM, False to load from disk each time
# ---------------------

# Get all image filenames (support common extensions)
all_images = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
# Optional quick subset during development
all_images = all_images[:len(all_images) // 4]
train_val_files, test_files = train_test_split(all_images, test_size=0.2, random_state=SEED)
train_files, val_files = train_test_split(train_val_files, test_size=0.2, random_state=SEED)

# Recommended transforms: keep boxes aligned by avoiding image resize unless boxes are updated too
data_transforms = transforms.Compose([
    transforms.ToTensor(),
])

# Choose Dataset class based on PRELOAD flag
if PRELOAD:
    print("Using PRELOADED mode: Images will be cached in RAM.")
    DatasetClass = PreloadedVegetableDataset
else:
    print("Using DISK mode: Images will be read from disk during training.")
    DatasetClass = VegetableDataset

# Create instances
train_dataset = DatasetClass(train_files, IMAGES_DIR, LABELS_DIR, data_transforms)
val_dataset = DatasetClass(val_files, IMAGES_DIR, LABELS_DIR, data_transforms)
test_dataset = VegetableDataset(test_files, IMAGES_DIR, LABELS_DIR, data_transforms)

# Collate function (essential for Object Detection to handle varying numbers of boxes per image)
def collate_fn(batch):
    return tuple(zip(*batch))

# Initialize DataLoaders using the BATCH_SIZE variable and safe NUM_WORKERS
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
)

print(f"\nDataLoaders ready.")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches.")

In [ ]:
def visualize_random_batch(dataloader, n_images=4):
    # Get a single batch
    images, targets = next(iter(dataloader))
    
    # Set up the plot
    fig, axes = plt.subplots(1, n_images, figsize=(20, 10))
    
    # Ensure axes is iterable even if n_images=1
    if n_images == 1:
        axes = [axes]
    
    for i in range(min(n_images, len(images))):
        # images[i] is a tensor [C, H, W] in [0, 1] range
        img_t = images[i].cpu().numpy()
        img = np.transpose(img_t, (1, 2, 0))
        img = np.clip(img, 0, 1)

        axes[i].imshow(img)
        axes[i].set_title(f"Image ID: {targets[i]['image_id'].item()}")
        
        # Get boxes and labels from the target dictionary
        boxes = targets[i]['boxes'].cpu().numpy()
        labels = targets[i]['labels'].cpu().numpy()
        
        for box, label in zip(boxes, labels):
            xmin, ymin, xmax, ymax = box
            width, height = xmax - xmin, ymax - ymin
            
            # Since we used label + 1 in the Dataset class for background, 
            # we subtract 1 here to match CLASS_NAMES keys.
            class_idx = int(label) - 1
            class_name = CLASS_NAMES.get(class_idx, f"ID: {class_idx}")
            
            # Draw the bounding box
            rect = patches.Rectangle(
                (xmin, ymin), width, height, 
                linewidth=2, edgecolor='lime', facecolor='none'
            )
            axes[i].add_patch(rect)
            
            # Add label text with a background box for readability
            axes[i].text(
                xmin, ymin - 5, class_name, 
                color='black', fontsize=12, fontweight='bold', 
                bbox=dict(facecolor='lime', alpha=0.7, pad=2)
            )
        
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Run the updated visualization
visualize_random_batch(train_loader)

In [ ]:
# 1. Update Model Configuration
NUM_OBJECT_CLASSES = len(CLASS_NAMES)  # number of object categories (no background)
NUM_MODEL_CLASSES = NUM_OBJECT_CLASSES + 1  # +1 for background

def get_model(num_model_classes):
    model = models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_model_classes)
    return model

model = get_model(NUM_MODEL_CLASSES)
model.to(device)

# 2. Advanced Optimizer and LR Scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# 3. Training Function with tqdm
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0.0
    
    # Create progress bar for the epoch
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc=f"Epoch {epoch} [Train]")
    
    for i, (images, targets) in pbar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        
        # Update progress bar with current batch loss
        pbar.set_postfix(loss=losses.item())

    avg_loss = total_loss / len(data_loader) if len(data_loader)>0 else float('inf')
    return avg_loss

@torch.no_grad()
def validate_one_epoch(model, data_loader, device):
    """Run validation. torchvision detection models return losses only when in train() mode, so we
    temporarily set train() while keeping gradients disabled with torch.no_grad(). Restore the
    original training state before returning."""
    was_training = model.training
    model.train()  # needed for loss computation in torchvision detectors

    val_loss = 0.0
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc="Validating")

    for i, (images, targets) in pbar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        val_loss += losses.item()

        pbar.set_postfix(val_loss=losses.item())

    # Restore original training/eval state
    if not was_training:
        model.eval()

    avg_val_loss = val_loss / len(data_loader) if len(data_loader) > 0 else float('inf')
    return avg_val_loss

# 4. Main Execution Loop
num_epochs = 2
best_model_wts = copy.deepcopy(model.state_dict())
best_loss = float('inf')

print(f"Starting Training on {device}...")
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss = validate_one_epoch(model, val_loader, device)

    print(f"Epoch {epoch} Summary: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    exp_lr_scheduler.step()

    if val_loss < best_loss:
        best_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, 'best_model.pth')
        print(f"*** Best model updated and saved to best_model.pth! (Val Loss: {best_loss:.4f}) ***")
    print("-" * 30)

model.load_state_dict(best_model_wts)
print("Training Complete. Best Validation Loss: {:.4f}".format(best_loss))

In [ ]:
# Visualize 3 test images with original boxes and model predictions
test_loader = DataLoader(
    test_dataset,
    batch_size=3,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
 )

def show_test_predictions(model, dataloader, device, score_threshold=0.5):
    model.eval()
    images, targets = next(iter(dataloader))

    with torch.no_grad():
        outputs = model([img.to(device) for img in images])

    fig, axes = plt.subplots(1, len(images), figsize=(18, 7))
    if len(images) == 1:
        axes = [axes]

    for i, (image, target, output) in enumerate(zip(images, targets, outputs)):
        img = image.permute(1, 2, 0).cpu().numpy()
        img = np.clip(img, 0, 1)

        axes[i].imshow(img)
        axes[i].set_title(f"Test Image {i + 1}")
        axes[i].axis('off')

        # Ground-truth boxes: green
        gt_boxes = target['boxes'].cpu().numpy()
        gt_labels = target['labels'].cpu().numpy()
        for box, label in zip(gt_boxes, gt_labels):
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='lime', facecolor='none'
            )
            axes[i].add_patch(rect)
            class_name = CLASS_NAMES.get(int(label) - 1, f'ID: {int(label) - 1}')
            axes[i].text(
                xmin, max(0, ymin - 5), f'GT: {class_name}',
                color='black', fontsize=10, fontweight='bold',
                bbox=dict(facecolor='lime', alpha=0.7, pad=2), clip_on=True
            )

        # Predicted boxes: red
        pred_boxes = output['boxes'].detach().cpu().numpy()
        pred_labels = output['labels'].detach().cpu().numpy()
        pred_scores = output['scores'].detach().cpu().numpy()

        for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
            if score < score_threshold:
                continue
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax - ymin,
                linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
            )
            axes[i].add_patch(rect)
            class_name = CLASS_NAMES.get(int(label) - 1, f'ID: {int(label) - 1}')
            axes[i].text(
                xmin, min(img.shape[0] - 1, ymax + 12), f'Pred: {class_name} {score:.2f}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(facecolor='red', alpha=0.75, pad=2), clip_on=True
            )

    plt.tight_layout(pad=0.5)
    plt.show()

# Run after training
show_test_predictions(model, test_loader, device, score_threshold=0.5)

In [ ]:
# --- Evaluation helpers: simple mAP@0.5 implementation ---

def box_iou(boxA, boxB):
    xa1, ya1, xa2, ya2 = boxA
    xb1, yb1, xb2, yb2 = boxB
    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    areaA = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    areaB = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def compute_ap(rec, prec):
    # Numerical integration (VOC style) of precision-recall curve
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return ap

def evaluate_map(model, dataloader, device, iou_threshold=0.5, score_threshold=0.05):
    model.eval()
    num_classes = NUM_MODEL_CLASSES
    cls_scores = {c: [] for c in range(1, num_classes)}
    cls_matches = {c: [] for c in range(1, num_classes)}
    cls_num_gts = {c: 0 for c in range(1, num_classes)}

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Evaluating"):
            images = list(img.to(device) for img in images)
            outputs = model(images)
            for i in range(len(outputs)):
                out = outputs[i]
                pred_boxes = out.get('boxes', torch.empty((0, 4))).cpu().numpy()
                pred_labels = out.get('labels', torch.empty((0,), dtype=torch.int64)).cpu().numpy()
                pred_scores = out.get('scores', torch.empty((0,))).cpu().numpy()
                gt_boxes = targets[i]['boxes'].cpu().numpy()
                gt_labels = targets[i]['labels'].cpu().numpy()
                for c in range(1, num_classes):
                    gt_mask = (gt_labels == c)
                    gt_boxes_c = gt_boxes[gt_mask]
                    cls_num_gts[c] += len(gt_boxes_c)
                    pred_mask = (pred_labels == c)
                    boxes_c = pred_boxes[pred_mask]
                    scores_c = pred_scores[pred_mask]
                    order = np.argsort(-scores_c)
                    boxes_c = boxes_c[order]
                    scores_c = scores_c[order]
                    matched = np.zeros(len(gt_boxes_c), dtype=bool)
                    for b, s in zip(boxes_c, scores_c):
                        if s < score_threshold:
                            continue
                        ious = np.array([box_iou(b, g) for g in gt_boxes_c]) if len(gt_boxes_c) > 0 else np.array([])
                        if ious.size > 0:
                            j = ious.argmax()
                            if ious[j] >= iou_threshold and not matched[j]:
                                cls_scores[c].append(s)
                                cls_matches[c].append(1)
                                matched[j] = True
                            else:
                                cls_scores[c].append(s)
                                cls_matches[c].append(0)
                        else:
                            cls_scores[c].append(s)
                            cls_matches[c].append(0)

    ap_per_class = {}
    for c in range(1, num_classes):
        scores = np.array(cls_scores[c])
        matches = np.array(cls_matches[c])
        npos = cls_num_gts[c]
        if npos == 0:
            ap_per_class[c] = None
            continue
        if scores.size == 0:
            ap_per_class[c] = 0.0
            continue
        order = np.argsort(-scores)
        matches = matches[order]
        tp = np.cumsum(matches == 1)
        fp = np.cumsum(matches == 0)
        rec = tp / npos
        prec = tp / (tp + fp + 1e-9)
        ap = compute_ap(rec, prec)
        ap_per_class[c] = ap

    valid_aps = [v for v in ap_per_class.values() if v is not None]
    mAP = float(np.mean(valid_aps)) if len(valid_aps) > 0 else 0.0
    return mAP, ap_per_class

# Run evaluation (set RUN_EVAL=True to execute; can be slow on large val sets)
RUN_EVAL = False
if RUN_EVAL:
    mAP, ap_per_class = evaluate_map(model, val_loader, device)
    print(f"mAP@0.5: {mAP:.4f}")
    for c, ap in ap_per_class.items():
        name = CLASS_NAMES.get(c - 1, str(c - 1))
        if ap is None:
            print(f"Class {c} ({name}): AP=N/A (no gt)")
        else:
            print(f"Class {c} ({name}): AP={ap:.4f}")

**Validation & Metrics (recommended)**
- **mAP@0.5**: Primary metric for object detection; mean average precision at IoU=0.5.
- **Precision / Recall**: Inspect trade-offs and choose score thresholds.
- **IoU**: Intersection over Union used for matching predictions to ground truth.
- **Train / Val / Test split**: Keep strictly separate to avoid leakage.
- **Reproducibility**: Fix seeds, record package versions, save checkpoints.
- **Baseline & Ablation**: Implement simple baselines and incremental changes.
- **Logging & Checkpoints**: Save best model and training logs (loss, lr, metrics).
- **Visualization**: Regularly inspect predictions and failure modes.
- **Compute Monitoring**: Track GPU memory and training time for reproducibility.
- **Documentation**: Describe dataset, preprocessing, label format, and evaluation protocol.